# N2 Phase A: development (Track C) - identity, detection, routing, device tiers
Runs PROTOCOL_A.md. Refuses to run if the embedded protocol differs from the signed hash.
Uses ODD attack instances only and the development split of TRAIN/CALIB days; EVEN instances and clean TEST are never computed.
Internet ON (downloads ~7 MB). CPU is enough (about 1-2 hours); GPU not used. Run all cells top to bottom.

In [ ]:
import json, hashlib, warnings
warnings.filterwarnings('ignore')
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
PROTOCOL_TEXT = "# N2 Phase A protocol (DEVELOPMENT, Track C): identity, attack detection, routing, device tiers\n\nExperiment N2, Phase A. Written 2026-11-08. Extends PREREG_N2.md (sha256 755fc38b...) and does not replace it.\nPhase A is exploration. Nothing here is a confirmed result. Confirmation is Phase B, which gets its own signed pre-registration and is run once.\n\n## 0. Declared history (read first)\n- The first core run (PREREG_N2) was already executed and its aggregate results were seen: identity positive (macro-F1 0.993), expert detection gate failed (E 0.566, alarm 0.128), routing rejected (E 0.566 vs G1 0.653). Amendment A1 (post hoc) was also seen, including per-instance detection files.\n- Therefore Phase A is a development phase on data whose aggregate behaviour is known. The confirmation in Phase B is strong but not pristine, and will say so.\n- No Phase A decision may use the outcome on even-numbered attack instances (section 2).\n\n## 1. Questions\nQ1 Which devices can be identified well enough to verify a claim (closed-set, all 10 devices)?\nQ2 Which detector family works, per device, without raising the clean-traffic alarm rate?\nQ3 Does routing a verified identity to a per-device detector beat the best global detector, family by family?\nQ4 Device tiers (section 8), which become the hypothesis of Phase B.\nTracks: Track C (this protocol and Phase B) is the only source of quoted results. Track F (refit on all data for the demo bundle) is built after Phase B and its numbers are never quoted.\n\n## 2. Data discipline\n- Instances of each device are sorted by start time and numbered from 1. ODD = DEV (used in Phase A). EVEN = CONFIRM (locked).\n- Code computes instance outcomes for DEV instances only. EVEN-instance outcomes are never computed in Phase A.\n- Attack neighbourhood (+/-2 h) minutes of ALL instances stay excluded from every training, calibration and clean block (unchanged from PREREG_N2).\n- Clean TEST days of PREREG_N2 are NOT opened in Phase A. Phase A uses a development split of the existing TRAIN and CALIB days:\n  DEV-TRAIN = first 75% of the device's TRAIN days; one-day gap; DEV-VAL = remaining TRAIN days; CALIB days unchanged (thresholds only).\n  Models are fitted on DEV-TRAIN, thresholds come from CALIB, clean alarm rate is measured on DEV-VAL.\n- A device is detection-ELIGIBLE if it has >=4 DEV instances.\n- Features: identical to PREREG_N2 section 3 (40 common features, detection input 80 values with trailing 5-minute mean, identity windows 15 min, 120 values).\n- Held-out identity devices: seed 20261108 as before (SamsungCamera, Chromecast), used only in the held-out identity run.\n\n## 3. Identity (Phase A.1)\nTwo runs, same code:\n(a) CLOSED-SET run: all 10 devices known. Used for per-device identity decisions (tiers).\n(b) HELD-OUT run: 2 devices unknown (as in PREREG_N2). Used for gates G3-G6 (impersonation, unknown devices).\nArms: RF(500), XGBoost(500, depth 6, lr 0.05), MLP (standardised input, hidden 128-64, adam, early stopping, minority classes oversampled to balance), each as single-window and vote over 3 consecutive windows. Seeds 11/22/33 (random_state of each model).\nPrimary arm: best macro-F1 on CALIB among admitted arms (seed-mean). Per-device claim threshold = 10th percentile of the true-claim probability on CALIB windows.\nMLP admission (deep-model rule): MLP is admitted only if its honest-acceptance macro (G2) exceeds the best RF/XGB arm by >=0.05 in EVERY seed, with G6 (false verification) not worse by more than 0.01. Otherwise it is reported as tried and not used. The reason an MLP is tried: Echo and Hue were weak under trees; a smooth model may separate them.\nPer-device identity decision (closed-set run, DEV-VAL, primary arm, seed-mean): identity SUPPORTED if honest acceptance >=0.80 AND per-device F1 >=0.80 AND random-mismatch rejection of claims for that device >=0.95.\nReported also: gates G1-G6 of PREREG_N2 on DEV-VAL (development, not confirmation), leaky 5-fold comparison, confusion matrix, same-cohort vs cross-cohort mismatch rejection.\n\n## 4. Detection (Phase A.2): families, modes, scopes\nEvery configuration = (family, mode, scope), run for seeds 11/22/33 separately.\nFamilies (the ladder; each rung must justify itself):\n- IF: IsolationForest(300 trees) on the 80-value input. Baseline, reproduces the failed first run on the development split.\n- AE: MLP autoencoder (80-64-16-64-80, relu, adam, batch 256, up to 60 epochs, stops early after 15 epochs without loss gain, MSE), inputs standardised on DEV-TRAIN and clipped to +/-20. Why: N-BaIoT / Kitsune method; models the whole feature vector.\n- TAE: same autoencoder on the lag-stack of the last 10 minutes of the 40 features (400 values). Why: tests whether time context matters, as the cheap test before any LSTM.\n- LSTM-AE: NOT run in Phase A. It is run in a separate sub-phase A.2b only if TAE beats AE under the ladder rule, because that would show temporal structure matters.\n- Transformer: not run. Enters only if LSTM-AE wins clearly in A.2b.\nScopes: E per-device expert (fitted on that device's DEV-TRAIN); G0 one global model (single threshold on pooled CALIB); G1 global model with the claimed device one-hot appended (per-device threshold).\nTraining caps (autoencoders only; IsolationForest uses all DEV-TRAIN minutes): at most 40,000 DEV-TRAIN minutes per device, 150,000 pooled (random subsample, seed-fixed). Standardisation: per device for E, pooled for G0/G1.\nModes:\n- static: threshold = 99th percentile of CALIB scores (G0: pooled; E, G1: per device).\n- rolling: threshold at minute t = median + k * 1.4826 * MAD of that device's scores over the preceding 4,320 rows (3 days; >=1,440 rows needed, otherwise the static threshold), shifted so only the past is used. k = smallest value in {1,2,3,4,5,6,8,10,12,15,20,30} giving <=1% alarms on the device's CALIB minutes (30 if none). Why: the first run failed from threshold drift (Netatmo, LIFX).\nInstance detected if >=2 alarm minutes lie inside it. Lag-stack rows before the first minute repeat the first row; sequences are by row order, not by clock time.\nAdmissible configuration: mean DEV-VAL alarm rate over eligible devices <=0.02.\nSupervised reference (not selectable): XGBoost, global with device one-hot, trained on DEV-TRAIN benign minutes plus DEV attack minutes of the OTHER attack families (leave-one-family-out), tested on DEV instances of the held-out family. Why: shows what labelled attacks add over benign-only detection. It runs with seed 11 only; alarm threshold = max(99th percentile of CALIB probabilities, 0.5). It cannot test routing and is not deployable as designed.\n\n## 5. Selection ladder (deep-model rule)\n1. Per family, the best admissible configuration = highest seed-mean macro DEV detection over eligible devices; ties go to the simpler one (static before rolling; G0 before G1 before E).\n2. Ladder IF -> AE -> TAE. A candidate replaces the incumbent only if in EVERY seed its macro DEV detection exceeds the incumbent's by >=0.05 and its mean DEV-VAL alarm rate is <= incumbent's + 0.005. If no IF configuration is admissible, the first admissible deep configuration becomes the incumbent and is reported as such.\n3. If no family has an admissible configuration, the detection verdict is \"no admissible detector\" and no device is detection-validated.\nThe configuration at the end of the ladder is SELECTED.\n\n## 6. Routing test (like-with-like, every family and mode)\nFor each (family, mode), E is compared with the better of G0 and G1 of the same family and mode: macro DEV detection difference, paired bootstrap over DEV instances (2,000 resamples, seed 20261108, seed-averaged hits), and alarm rates.\nRouting passes for that (family, mode) if difference >=+0.05 in every seed, bootstrap lower 95% bound >0, and E alarm <= global alarm + 0.005.\nRouting is KEPT only if the SELECTED configuration has scope E and passes. Otherwise routing is REMOVED, the ladder of section 5 is re-run over global scopes (G0, G1) only and its result is the detector, and the report states \"routing tried across families and failed\" with the table. Per-family outcomes are reported either way.\n\n## 7. Mismatch states\nUnchanged from PREREG_N2 section 8 (MATCH, MISMATCH, UNKNOWN, UNCERTAIN, UNREGISTERED_CLAIM, INPUT_REJECTED).\n\n## 8. Device tiers (output of Phase A, hypothesis of Phase B)\n- VERIFIED: identity supported (section 3) AND detection supported by the selected configuration: seed-mean DEV detection >=0.80, seed-mean DEV-VAL alarm <=0.02, >=4 DEV instances.\n- IDENTIFIED_UNVALIDATED: identity supported, detection not supported. Monitor-only: the system can say \"detected but not confident enough\"; it makes no detection claim.\n- UNREGISTERED: identity not supported. Claims for it cannot be verified.\nTiers are assigned from DEV data only. Phase B confirms them on EVEN instances and the locked clean TEST.\n\n## 9. Run rules\n- Trials: every run is logged in RUN_LOG with what changed and why. At most 3 logged iterations of settings (hyper-parameters fixed above; changes must be justified by a failure shown in the logs, e.g. non-convergence).\n- Bug fixes are logged and do not count as iterations.\n- No peeking: the notebook cannot print EVEN-instance outcomes or clean-TEST results by construction.\n- Negative results are reported as negative.\n\n## 10. Known weaknesses stated in advance\n- Aggregate results of the first run are known (section 0).\n- Half the attack instances per device: iHome (3 instances, 2 DEV) and Echo (9, 5 DEV) are marginal.\n- Same lab, scripted attacks, 10 devices, two recording periods: cohort effects may help identity.\n- Autoencoder training uses a random subsample and 60 epochs without tuning.\n- One dataset in one lab does not establish general IoT security.\n\n## 11. Freeze\nThis file's SHA-256 is signed with the N2 key as block 2 of the N2 chain before the Phase A notebook is run. The notebook embeds this text and refuses to run if its hash differs.\n"
CHAIN_BLOCK = "{\"hash\": \"659469f4a5a5070b210813fe6af797eb90899a0787dfafaed6c2733891f5ea0f\", \"header\": {\"index\": 2, \"merkle_root\": \"2310adffe6b328636db7b69dafdd3435483b74511957044c1221827e80960654\", \"prev_hash\": \"8ac5656d09b67e42ccedc5585dd6c6cfe721f4530d51c54c8ee301c00a1c3101\", \"signer_pub\": \"93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23\", \"timestamp\": 1794159553.008113}, \"payload\": {\"experiment\": \"N2\", \"file\": \"PROTOCOL_A.md\", \"kind\": \"protocol_freeze\", \"note\": \"Phase A development protocol; signed before the Phase A notebook is run; odd instances only\", \"phase\": \"A\", \"sha256\": \"b4f61d31282cb0ba14333804d6391048d8dc130dde0d5459c8558bb3bdc65ec8\"}, \"sig\": \"6815eaa5a17529f3c46481643dd0d8dc37b61d87911f83175e4448cd42ead5336910e3afd77e1f0ec60a41b5947ddbbba4a11e8dacb17b20eeac1e1af9f5e405\"}"
PUB = '93d2db7c37ec10acb6e189b61f6a5eacdddb74afd482d9e5bbcd3c41e1213b23'
def _canon(o): return json.dumps(o, sort_keys=True, separators=(',', ':')).encode()
blk = json.loads(CHAIN_BLOCK)
sha = hashlib.sha256(PROTOCOL_TEXT.encode()).hexdigest()
assert blk['payload']['sha256'] == sha, 'PROTOCOL text differs from the signed version: STOP'
assert blk['payload']['kind'] == 'protocol_freeze' and blk['header']['signer_pub'] == PUB
assert hashlib.sha256(_canon({'header': blk['header'], 'payload': blk['payload']})).hexdigest() == blk['hash'], 'block hash mismatch'
Ed25519PublicKey.from_public_bytes(bytes.fromhex(PUB)).verify(bytes.fromhex(blk['sig']), blk['hash'].encode())
print('PROTOCOL_A signature OK | sha256', sha, '| chain block', blk['header']['index'], blk['hash'])

## Libraries (embedded; same files that passed the synthetic smoke test)

In [ ]:
"""N2 core: identity + per-device attack detection + routing on the UNSW attack dataset.
Implements PREREG_N2.md exactly. Interpretations the prereg left open are marked INTERPRETATION."""
import os, re, json, glob, hashlib
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

DEVICES = {'ec1a59832811': ('WeMoMotion', 1), 'ec1a5979f489': ('WeMoSwitch', 1), '00166cab6b88': ('SamsungCamera', 1),
           '50c7bf005639': ('TPLinkPlug', 1), '70ee50183443': ('NetatmoCamera', 1),
           '0017882b9a25': ('HueBulb', 2), '44650d56ccd3': ('Echo', 2), 'f4f5d88f0a3c': ('Chromecast', 2),
           '74c63b29d71d': ('iHome', 2), 'd073d5018308': ('LIFX', 2)}
CFG = dict(split_seed=20261108, seeds=(11, 22, 33), if_trees=300, rf_trees=500, xgb_trees=500,
           boot=1000, boot_inst=2000, min_attack_inst=5)
COL_RE = re.compile(r'^(From|To)(Local|Internet)(.+?)(Packet|Byte)(?:\.\d+)?$')
FEATS = [f'{d}{l}_{p}_{u}' for d in ('From', 'To') for l in ('Local', 'Internet')
         for p in ('Tcp', 'Udp', 'Icmp', 'Arp', 'Other') for u in ('Packet', 'Byte')]

def proto_class(mid):
    for k in ('Tcp', 'Udp', 'Icmp', 'Arp'):
        if mid.startswith(k): return k
    return 'Other'

def common_features(df):
    groups, unmatched = {}, []
    for c in df.columns:
        if c == 'Timestamp': continue
        m = COL_RE.match(str(c))
        if not m: unmatched.append(c); continue
        d, l, mid, u = m.groups()
        groups.setdefault(f'{d}{l}_{proto_class(mid)}_{u}', []).append(c)
    X = pd.DataFrame(0.0, index=df.index, columns=FEATS)
    for k, cl in groups.items():
        X[k] = df[cl].apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0).sum(axis=1)
    other_cols = sum(len(v) for k, v in groups.items() if '_Other_' in k)
    return np.log1p(X), unmatched, other_cols

def minute_table(df):
    df = df.sort_values('Timestamp').reset_index(drop=True)
    X, unm, oc = common_features(df)
    M = X.rolling(5, min_periods=1).mean()
    Xn = X.to_numpy(np.float32)
    return dict(ts=df['Timestamp'].astype('int64').to_numpy(), X=Xn,
                F=np.hstack([Xn, M.to_numpy(np.float32)]), unmatched=unm, other_cols=oc)

ATK_RE = re.compile(r'^(.*?)(\d+)([LWD]2[LWD])$')
def parse_annotations(path):
    inst = []
    for line in open(path, errors='replace'):
        line = line.strip()
        if not line: continue
        h = line.split(',', 2)
        if len(h) < 3: raise ValueError(f'unparseable annotation line in {path}: {line[:80]}')
        s, e = float(h[0]), float(h[1])            # raises if a header sneaks in: STOP rule
        name = h[2].rpartition(',')[2].strip()
        inst.append((s, e, name))
    inst = sorted(set(inst))
    if not inst: raise ValueError(f'no instances in {path}')
    return inst

def attack_parts(name):
    m = ATK_RE.match(name)
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (name, -1, '')

def split_days(ts, inst, min_rows=600):
    day = (ts // 86400000).astype(int)
    neigh = np.zeros(len(ts), bool)
    for s, e, _ in inst:
        neigh |= (ts >= (s - 7200) * 1000) & (ts <= (e + 7200) * 1000)
    ad = set(day[neigh].tolist()); adb = ad | {d - 1 for d in ad} | {d + 1 for d in ad}
    u, c = np.unique(day, return_counts=True)
    clean = [int(d) for d, k in zip(u, c) if k >= min_rows and int(d) not in adb]
    n = len(clean)
    if n < 10: raise ValueError(f'only {n} clean days')
    a = int(0.70 * n); b = a + 1 + int(0.15 * n)
    return day, neigh, dict(train=set(clean[:a]), calib=set(clean[a + 1:b]), test=set(clean[b + 1:])), n

def load_all(flow_dir, ann_dir):
    D = {}
    for mac, (name, coh) in DEVICES.items():
        fp = glob.glob(f'{flow_dir}/**/{mac}_flowstats.csv', recursive=True)
        ap = glob.glob(f'{ann_dir}/**/{mac}.csv', recursive=True)
        if not fp or not ap: raise FileNotFoundError(f'{mac}: flow={bool(fp)} annotation={bool(ap)}')
        mt = minute_table(pd.read_csv(fp[0], low_memory=False))
        inst = parse_annotations(ap[0])
        day, neigh, sp, n = split_days(mt['ts'], inst)
        d = dict(name=name, cohort=coh, inst=inst, day=day, neigh=neigh, split=sp, n_clean=n, **mt)
        for k in ('train', 'calib', 'test'):
            d['m_' + k] = np.isin(day, list(sp[k])) & ~neigh
        D[mac] = d
    return D

# ---------------- detection ----------------
def fit_if(X, seeds, trees):
    return [IsolationForest(n_estimators=trees, random_state=s, n_jobs=-1).fit(X) for s in seeds]
def score_if(models, X):
    return np.mean([-m.score_samples(X) for m in models], axis=0)
def inst_hits(ts, sc, thr, inst, min_alarms=2):
    return np.array([int(((sc[(ts >= s * 1000) & (ts <= e * 1000)]) >= thr).sum() >= min_alarms) for s, e, _ in inst])

def run_detection(D, cfg):
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}
    oh = lambda m, n: np.tile(np.eye(len(macs), dtype=np.float32)[idx[m]], (n, 1))
    E = {m: fit_if(D[m]['F'][D[m]['m_train']], cfg['seeds'], cfg['if_trees']) for m in macs}
    G0 = fit_if(np.vstack([D[m]['F'][D[m]['m_train']] for m in macs]), cfg['seeds'], cfg['if_trees'])
    G1 = fit_if(np.vstack([np.hstack([D[m]['F'][D[m]['m_train']], oh(m, int(D[m]['m_train'].sum()))]) for m in macs]),
                cfg['seeds'], cfg['if_trees'])
    sE, s0, s1 = {}, {}, {}
    for m in macs:
        F = D[m]['F']; n = len(F)
        sE[m] = score_if(E[m], F); s0[m] = score_if(G0, F); s1[m] = score_if(G1, np.hstack([F, oh(m, n)]))
    thrE = {m: np.quantile(sE[m][D[m]['m_calib']], 0.99) for m in macs}
    thr0 = np.quantile(np.concatenate([s0[m][D[m]['m_calib']] for m in macs]), 0.99)
    thr1 = {m: np.quantile(s1[m][D[m]['m_calib']], 0.99) for m in macs}
    rows, per_inst = [], []
    for m in macs:
        d = D[m]; ts = d['ts']; te = d['m_test']
        hE = inst_hits(ts, sE[m], thrE[m], d['inst']); h0 = inst_hits(ts, s0[m], thr0, d['inst']); h1 = inst_hits(ts, s1[m], thr1[m], d['inst'])
        rows.append(dict(mac=m, device=d['name'], cohort=d['cohort'], n_inst=len(d['inst']), eligible=len(d['inst']) >= cfg['min_attack_inst'],
                         det_E=hE.mean(), det_G0=h0.mean(), det_G1=h1.mean(),
                         alarm_E=float((sE[m][te] >= thrE[m]).mean()), alarm_G0=float((s0[m][te] >= thr0).mean()), alarm_G1=float((s1[m][te] >= thr1[m]).mean())))
        for k, (s, e, nm) in enumerate(d['inst']):
            f, r, p = attack_parts(nm)
            per_inst.append(dict(mac=m, device=d['name'], name=nm, family=f, rate=r, path=p, hit_E=int(hE[k]), hit_G0=int(h0[k]), hit_G1=int(h1[k]),
                                 n_rows=int(((ts >= s * 1000) & (ts <= e * 1000)).sum())))
    det = pd.DataFrame(rows); pi = pd.DataFrame(per_inst)
    # wrong-expert: expert of A applied to device B's attack traffic
    wrong = []
    for B in macs:
        dB = D[B]
        for A in macs:
            if A == B: continue
            sc = score_if(E[A], dB['F'])
            wrong.append(dict(claimed=D[A]['name'], actual=dB['name'], same_cohort=D[A]['cohort'] == dB['cohort'],
                              det=float(inst_hits(dB['ts'], sc, thrE[A], dB['inst']).mean()),
                              eligible=len(dB['inst']) >= cfg['min_attack_inst']))
    return det, pi, pd.DataFrame(wrong)

def macro(det, col, elig=True):
    t = det[det.eligible] if elig else det
    return float(t[col].mean())

def routing_verdict(det, pi, cfg, seed=20261108):
    el = det[det.eligible]
    mE, m0, m1 = macro(det, 'det_E'), macro(det, 'det_G0'), macro(det, 'det_G1')
    best = 'G0' if m0 >= m1 else 'G1'; mG = max(m0, m1)
    aE = macro(det, 'alarm_E'); aG = macro(det, 'alarm_' + best)
    rng = np.random.default_rng(seed); macs = el.mac.tolist()
    hits = {m: pi[pi.mac == m][['hit_E', 'hit_' + best]].to_numpy() for m in macs}
    diffs = []
    for _ in range(cfg['boot_inst']):
        dd = []
        for m in macs:
            h = hits[m]; i = rng.integers(0, len(h), len(h)); dd.append(h[i, 0].mean() - h[i, 1].mean())
        diffs.append(np.mean(dd))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    accepted = bool((mE - mG) >= 0.05 and lo > 0 and aE <= aG + 0.005)
    gate_A1 = bool(mE >= 0.80 and aE <= 0.02)
    return dict(det_E=mE, det_G0=m0, det_G1=m1, best_global=best, delta=mE - mG, delta_ci95=[float(lo), float(hi)],
                alarm_E=aE, alarm_best_global=aG, gate_A1_expert=gate_A1,
                routing='routing_accepted' if accepted else 'routing_rejected_use_global', eligible_devices=el.device.tolist())

# ---------------- identity ----------------
def windows(d, days, win_ms=15 * 60000, min_rows=10):
    sel = np.isin(d['day'], list(days)) & ~d['neigh']
    if not sel.any(): return np.zeros((0, 120), np.float32), np.zeros(0, int)
    t = d['ts'][sel]; x = d['X'][sel]; df = pd.DataFrame(x); df['w'] = t // win_ms; df['day'] = d['day'][sel]
    g = df.groupby('w'); keep = g.size(); keep = keep[keep >= min_rows].index
    cols = list(range(x.shape[1]))
    agg = g[cols].agg(['mean', 'std', 'max']).loc[keep]
    F = agg.to_numpy(np.float32); F = np.nan_to_num(F)
    return F, g['day'].first().loc[keep].to_numpy()

def vote(P, k=3):
    if len(P) == 0: return P
    return pd.DataFrame(P).rolling(k, min_periods=1).mean().to_numpy()

def make_model(kind, cfg, n_classes):
    if kind == 'RF':
        return RandomForestClassifier(n_estimators=cfg['rf_trees'], class_weight='balanced', n_jobs=-1, random_state=11)
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=cfg['xgb_trees'], max_depth=6, learning_rate=0.05, objective='multi:softprob',
                         tree_method='hist', n_jobs=-1, random_state=11, num_class=n_classes)

def pick_unknown(D, seed):
    rng = np.random.default_rng(seed); out = []
    for coh in (1, 2):
        ms = sorted(m for m in D if D[m]['cohort'] == coh); out.append(ms[int(rng.integers(0, len(ms)))])
    return out

def cluster_boot(keys, vals, B, seed):
    keys = np.asarray(keys); vals = np.asarray(vals, float)
    u, inv = np.unique(keys, return_inverse=True)
    s = np.bincount(inv, vals); c = np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        i = rng.integers(0, len(u), len(u)); out.append(s[i].sum() / c[i].sum())
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

def state_of(P_row, claim, thr, known_idx):
    if claim is None: return 'UNREGISTERED_CLAIM'
    if P_row[claim] >= thr[claim]: return 'MATCH'
    pm = P_row.max()
    return 'MISMATCH' if pm >= 0.60 else ('UNKNOWN' if pm < 0.35 else 'UNCERTAIN')

def run_identity(D, cfg):
    unk = pick_unknown(D, cfg['split_seed']); known = sorted(m for m in D if m not in unk)
    kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    W = {}
    for m in D:
        for part in ('train', 'calib', 'test'):
            W[(m, part)] = windows(D[m], D[m]['split'][part])
            if len(W[(m, part)][0]) == 0:   # STOP rule: report, do not silently drop a device
                raise ValueError(f'no identity windows for {D[m]["name"]} in {part}')
    def stack(part, devs):
        Xs, ys, ds, dv = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); ds += list(dy); dv += [m] * len(F)
        return np.vstack(Xs), np.array(ys), np.array(ds), np.array(dv)
    Xtr, ytr, _, _ = stack('train', known); Xca, yca, dca, vca = stack('calib', known); Xte, yte, dte, vte = stack('test', known)
    Xun, _, dun, vun = stack('test', unk)
    arms = {}
    for kind in ('RF', 'XGB'):
        mdl = make_model(kind, cfg, K)
        if kind == 'XGB': mdl.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else: mdl.fit(Xtr, ytr)
        P = {n: mdl.predict_proba(X) for n, X in (('ca', Xca), ('te', Xte), ('un', Xun))}
        def per_dev_vote(P_, vv):
            out = P_.copy()
            for m in np.unique(vv):
                ii = np.where(vv == m)[0]; out[ii] = vote(P_[ii])
            return out
        for mode in ('single', 'vote3'):
            Q = P if mode == 'single' else {'ca': per_dev_vote(P['ca'], vca), 'te': per_dev_vote(P['te'], vte), 'un': per_dev_vote(P['un'], vun)}
            arms[f'{kind}-{mode}'] = dict(P=Q, f1_calib=float(f1_score(yca, Q['ca'].argmax(1), average='macro')))
    order = ['RF-single', 'RF-vote3', 'XGB-single', 'XGB-vote3']
    primary = max(order, key=lambda a: (round(arms[a]['f1_calib'], 12), -order.index(a)))
    Pa = arms[primary]['P']
    thr = np.array([np.quantile(Pa['ca'][yca == k, k], 0.10) for k in range(K)])
    pred = Pa['te'].argmax(1); res = dict(primary=primary, calib_macro_f1={a: arms[a]['f1_calib'] for a in order}, unknown_devices=unk, known_devices=known)
    G1 = float(f1_score(yte, pred, average='macro'))
    acc = (Pa['te'][np.arange(len(yte)), yte] >= thr[yte]).astype(float)
    per_dev_acc = {known[k]: float(acc[yte == k].mean()) for k in range(K)}
    G2 = float(np.mean(list(per_dev_acc.values())))              # INTERPRETATION: macro over known devices
    rng = np.random.default_rng(cfg['split_seed'])
    claim_r = np.array([rng.choice([k for k in range(K) if k != y]) for y in yte])
    rej_r = (Pa['te'][np.arange(len(yte)), claim_r] < thr[claim_r]).astype(float)
    hard = {}
    for A in range(K):
        mp = {B: Pa['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
    rows_h = [(i, A) for A in range(K) for i in np.where(yte == hard[A])[0]]
    rej_h = np.array([float(Pa['te'][i, A] < thr[A]) for i, A in rows_h]) if rows_h else np.array([1.0])
    key_h = [f'{vte[i]}_{dte[i]}' for i, A in rows_h]
    rej_u = np.array([[float(Pa['un'][i, A] < thr[A]) for A in range(K)] for i in range(len(Xun))]) if len(Xun) else np.zeros((1, K))
    rej_u_flat = rej_u.ravel(); key_u = np.repeat([f'{vun[i]}_{dun[i]}' for i in range(len(Xun))], K) if len(Xun) else ['x']
    G3 = float(rej_r.mean()); G4 = float(rej_h.mean()); G5 = float(rej_u_flat.mean())
    G6 = float(1 - np.concatenate([rej_r, rej_h, rej_u_flat]).mean())
    B = cfg['boot']; kt = [f'{vte[i]}_{dte[i]}' for i in range(len(yte))]
    ci = dict(G1=cluster_boot(kt, (pred == yte).astype(float), B, 1)[:], G2=cluster_boot(kt, acc, B, 2), G3=cluster_boot(kt, rej_r, B, 3),
              G4=cluster_boot(key_h, rej_h, B, 4), G5=cluster_boot(key_u, rej_u_flat, B, 5))
    ci['G1'] = 'accuracy CI shown (macro-F1 CI not bootstrapped)'; ci['note'] = 'cluster bootstrap over device-days'
    gates = dict(G1=G1 >= 0.80, G2=G2 >= 0.80, G3=G3 >= 0.95, G4=G4 >= 0.90, G5=G5 >= 0.60, G6=G6 <= 0.10)
    nf = sum(not v for k, v in gates.items() if k != 'G1')
    verdict = 'POSITIVE' if all(gates.values()) else ('PARTIAL' if gates['G1'] and nf <= 1 else 'NEGATIVE')
    # leaky comparison
    allX = np.vstack([Xtr, Xca, Xte]); ally = np.concatenate([ytr, yca, yte]); fl = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(allX, ally):
        mdl = make_model(primary.split('-')[0], cfg, K)
        (mdl.fit(allX[a], ally[a], sample_weight=compute_sample_weight('balanced', ally[a])) if primary.startswith('XGB') else mdl.fit(allX[a], ally[a]))
        fl.append(f1_score(ally[b], mdl.predict(allX[b]), average='macro'))
    # states
    def states(P_rows, claims):
        return pd.Series([state_of(P_rows[i], claims[i], thr, kidx) for i in range(len(P_rows))]).value_counts(normalize=True).round(4).to_dict()
    S = dict(honest=states(Pa['te'], list(yte)), random_mismatch=states(Pa['te'], list(claim_r)),
             hard_mismatch=states(Pa['te'][[i for i, A in rows_h]], [A for i, A in rows_h]) if rows_h else {},
             unknown_device=states(np.repeat(Pa['un'], K, axis=0), list(np.tile(np.arange(K), len(Xun)))) if len(Xun) else {})
    confusion = pd.crosstab(pd.Series([known[i] for i in yte], name='actual'), pd.Series([known[i] for i in pred], name='pred'))
    res.update(G1_macro_f1=G1, G2_honest_acceptance_macro=G2, G2_micro=float(acc.mean()), G3_random_mismatch_rejection=G3,
               G4_hard_mismatch_rejection=G4, G5_unknown_rejection=G5, G6_false_verification=G6, gates=gates, verdict=verdict,
               per_device_acceptance=per_dev_acc, hard_partner={known[A]: known[hard[A]] for A in range(K)}, ci95=ci,
               leaky_random_5fold_macro_f1=float(np.mean(fl)), states=S, n_windows=dict(train=len(Xtr), calib=len(Xca), test=len(Xte), unknown_test=len(Xun)))
    return res, confusion

In [ ]:
"""N2 Phase A (development). Implements PROTOCOL_A.md. Requires n2_lib.py (same notebook) for loading and features.
Outcomes on EVEN-numbered attack instances and on clean TEST days are never computed here (d['dev_inst'] holds the ODD ones only)."""
import warnings
import numpy as np, pandas as pd
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
try:                                   # in the notebook these names already exist (n2_lib is pasted above)
    from n2_lib import attack_parts, windows, vote
except ImportError:
    pass

ACFG = dict(seeds=(11, 22, 33), ae_epochs=60, ae_hidden=(64, 16, 64), tae_hidden=(128, 32, 128), tae_lags=10,
            cap_dev=40000, cap_pool=150000, min_dev_inst=4, clip=20.0,
            roll_win=4320, roll_min=1440, k_grid=(1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30),
            alarm_adm=0.02, det_margin=0.05, alarm_slack=0.005, boot_inst=2000, boot_seed=20261108)
FAMILIES = ('IF', 'AE', 'TAE'); MODES = ('static', 'rolling'); SCOPES = ('G0', 'G1', 'E')

# ---------------- development split ----------------
def prepare_dev(D, acfg=ACFG):
    for m, d in D.items():
        tr = sorted(d['split']['train']); a = int(0.75 * len(tr))
        dtr, dval = set(tr[:a]), set(tr[a + 1:])
        d['dev_days'] = (dtr, dval)
        d['m_dtrain'] = np.isin(d['day'], list(dtr)) & ~d['neigh']
        d['m_dval'] = np.isin(d['day'], list(dval)) & ~d['neigh']
        d['dev_inst'] = d['inst'][0::2]                     # ODD instances (1st, 3rd, ...) = DEV
        d['eligible'] = len(d['dev_inst']) >= acfg['min_dev_inst']
        if d['m_dtrain'].sum() < 500 or d['m_dval'].sum() < 100:
            raise ValueError(f"{d['name']}: development split too small (train {d['m_dtrain'].sum()}, val {d['m_dval'].sum()})")
        d.pop('m_test', None)                                 # clean TEST mask removed: cannot be used by mistake
    return D

# ---------------- detectors ----------------
def lagstack(X, L):
    n = len(X); out = np.empty((n, X.shape[1] * L), np.float32)
    for j in range(L):
        idx = np.maximum(np.arange(n) - j, 0)
        out[:, j * X.shape[1]:(j + 1) * X.shape[1]] = X[idx]
    return out

class AE:
    def __init__(self, hidden, epochs, seed, clip):
        self.h, self.ep, self.seed, self.clip = hidden, epochs, seed, clip
    def _z(self, X): return np.clip(self.sc.transform(X), -self.clip, self.clip)
    def fit(self, X):
        self.sc = StandardScaler().fit(X); Z = self._z(X)
        self.m = MLPRegressor(hidden_layer_sizes=self.h, activation='relu', solver='adam', max_iter=self.ep, batch_size=256,
                              learning_rate_init=1e-3, tol=1e-6, n_iter_no_change=15, random_state=self.seed).fit(Z, Z)
        return self
    def score(self, X):
        out = np.empty(len(X))
        for i in range(0, len(X), 50000):
            Z = self._z(X[i:i + 50000]); out[i:i + 50000] = ((self.m.predict(Z) - Z) ** 2).mean(1)
        return out

class IFW:
    def __init__(self, trees, seed): self.t, self.s = trees, seed
    def fit(self, X):
        from sklearn.ensemble import IsolationForest
        self.m = IsolationForest(n_estimators=self.t, random_state=self.s, n_jobs=-1).fit(X); return self
    def score(self, X): return -self.m.score_samples(X)

def rolling_thr(s, calib, static_thr, acfg):
    ser = pd.Series(s)
    med = ser.rolling(acfg['roll_win'], min_periods=acfg['roll_min']).median().shift(1)
    mad = (ser - med).abs().rolling(acfg['roll_win'], min_periods=acfg['roll_min']).median().shift(1)
    best = None
    for k in acfg['k_grid']:
        thr = (med + k * 1.4826 * mad).to_numpy(); thr = np.where(np.isnan(thr), static_thr, thr)
        best = (k, thr)
        if (s[calib] >= thr[calib]).mean() <= 0.01: break
    return best

def hits_arr(ts, alarm, inst):
    out = []
    for s, e, _ in inst:
        lo = np.searchsorted(ts, s * 1000, 'left'); hi = np.searchsorted(ts, e * 1000, 'right')
        out.append(int(alarm[lo:hi].sum() >= 2))
    return np.array(out, int)

def run_family(D, family, seed, acfg=ACFG, cfg=None):
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}; eye = np.eye(len(macs), dtype=np.float32)
    rng = np.random.default_rng(seed)
    def Xin(m):
        d = D[m]; return d['F'] if family != 'TAE' else lagstack(d['X'], acfg['tae_lags'])
    def new():
        if family == 'IF': return IFW(cfg['if_trees'], seed)
        return AE(acfg['ae_hidden'] if family == 'AE' else acfg['tae_hidden'], acfg['ae_epochs'], seed, acfg['clip'])
    def samp(m, n):
        ii = np.where(D[m]['m_dtrain'])[0]
        if family != 'IF' and len(ii) > n: ii = np.sort(rng.choice(ii, n, replace=False))
        return ii
    X = {m: Xin(m) for m in macs}
    E = {m: new().fit(X[m][samp(m, acfg['cap_dev'])]) for m in macs}
    per = acfg['cap_pool'] // len(macs)
    Xp = {m: X[m][samp(m, per)] for m in macs}
    G0 = new().fit(np.vstack([Xp[m] for m in macs]))
    G1 = new().fit(np.vstack([np.hstack([Xp[m], np.tile(eye[idx[m]], (len(Xp[m]), 1))]) for m in macs]))
    S = {'E': {}, 'G0': {}, 'G1': {}}
    for m in macs:
        S['E'][m] = E[m].score(X[m]); S['G0'][m] = G0.score(X[m])
        S['G1'][m] = G1.score(np.hstack([X[m], np.tile(eye[idx[m]], (len(X[m]), 1))]))
    del X, Xp
    thr0 = np.quantile(np.concatenate([S['G0'][m][D[m]['m_calib']] for m in macs]), 0.99)
    rows, prow = [], []
    for scope in SCOPES:
        for m in macs:
            d = D[m]; s = S[scope][m]
            st = thr0 if scope == 'G0' else np.quantile(s[d['m_calib']], 0.99)
            rk, rthr = rolling_thr(s, d['m_calib'], st, acfg)
            for mode, thr, k in (('static', np.full(len(s), st), np.nan), ('rolling', rthr, rk)):
                al = s >= thr; h = hits_arr(d['ts'], al, d['dev_inst'])
                rows.append(dict(family=family, mode=mode, scope=scope, seed=seed, mac=m, device=d['name'], eligible=d['eligible'],
                                 n_dev_inst=len(d['dev_inst']), det=float(h.mean()), alarm_val=float(al[d['m_dval']].mean()), k=k))
                for j, v in enumerate(h): prow.append((family, mode, scope, seed, m, j, int(v)))
    pi = pd.DataFrame(prow, columns=['family', 'mode', 'scope', 'seed', 'mac', 'inst', 'hit'])
    return pd.DataFrame(rows), pi

# ---------------- supervised reference (leave one attack family out) ----------------
def run_supervised(D, seed, acfg=ACFG, cfg=None):
    from xgboost import XGBClassifier
    macs = list(D); idx = {m: i for i, m in enumerate(macs)}; eye = np.eye(len(macs), dtype=np.float32)
    rng = np.random.default_rng(seed)
    def oh(m, n): return np.tile(eye[idx[m]], (n, 1))
    ben = {}
    for m in macs:
        ii = np.where(D[m]['m_dtrain'])[0]
        if len(ii) > acfg['cap_pool'] // len(macs): ii = np.sort(rng.choice(ii, acfg['cap_pool'] // len(macs), replace=False))
        ben[m] = ii
    atk = []                                                     # (mac, family, row indices)
    for m in macs:
        d = D[m]
        for s, e, nm in d['dev_inst']:
            lo = np.searchsorted(d['ts'], s * 1000, 'left'); hi = np.searchsorted(d['ts'], e * 1000, 'right')
            atk.append((m, attack_parts(nm)[0], np.arange(lo, hi), nm))
    fams = sorted({a[1] for a in atk}); out, alarms = [], {m: [] for m in macs}
    if len(fams) < 2: return pd.DataFrame(), 'single family: leave-one-family-out undefined'
    for f in fams:
        Xb = np.vstack([np.hstack([D[m]['F'][ben[m]], oh(m, len(ben[m]))]) for m in macs]); yb = np.zeros(len(Xb))
        tr = [a for a in atk if a[1] != f and len(a[2])]
        if not tr: continue
        Xa = np.vstack([np.hstack([D[a[0]]['F'][a[2]], oh(a[0], len(a[2]))]) for a in tr]); ya = np.ones(len(Xa))
        mdl = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method='hist', n_jobs=-1, random_state=seed)
        w = np.concatenate([np.ones(len(Xb)), np.full(len(Xa), max(1.0, len(Xb) / max(len(Xa), 1) / 5))])
        mdl.fit(np.vstack([Xb, Xa]), np.concatenate([yb, ya]), sample_weight=w)
        for m in macs:
            d = D[m]; p = mdl.predict_proba(np.hstack([d['F'], oh(m, len(d['F']))]))[:, 1]
            thr = max(np.quantile(p[d['m_calib']], 0.99), 0.5); al = p >= thr; alarms[m].append(float(al[d['m_dval']].mean()))
            for a in [a for a in atk if a[0] == m and a[1] == f]:
                out.append(dict(mac=m, device=d['name'], family=f, inst=a[3], hit=int(al[a[2]].sum() >= 2)))
    r = pd.DataFrame(out)
    summ = r.groupby(['mac', 'device']).hit.agg(['mean', 'count']).reset_index().rename(columns={'mean': 'det_lofo', 'count': 'n_inst'})
    summ['alarm_val'] = summ.mac.map({m: float(np.mean(v)) if v else np.nan for m, v in alarms.items()})
    summ['eligible'] = summ.mac.map({m: D[m]['eligible'] for m in macs})
    return summ, r

# ---------------- selection ladder and routing ----------------
def cfg_table(dev):
    el = dev[dev.eligible]
    g = el.groupby(['family', 'mode', 'scope', 'seed']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean')).reset_index()
    rows = []
    for (f, m, s), t in g.groupby(['family', 'mode', 'scope']):
        t = t.sort_values('seed')
        rows.append(dict(family=f, mode=m, scope=s, det_by_seed=t.det.tolist(), alarm_by_seed=t.alarm.tolist(),
                         det_mean=float(t.det.mean()), alarm_mean=float(t.alarm.mean()), admissible=bool(t.alarm.mean() <= ACFG['alarm_adm'])))
    return pd.DataFrame(rows)

def _best(ct, family, scopes):
    t = ct[(ct.family == family) & ct.admissible & ct.scope.isin(scopes)].copy()
    if t.empty: return None
    t['mr'] = t['mode'].map({'static': 0, 'rolling': 1}); t['sr'] = t.scope.map({'G0': 0, 'G1': 1, 'E': 2}); t['d'] = t.det_mean.round(12)
    return t.sort_values(['d', 'mr', 'sr'], ascending=[False, True, True]).iloc[0]

def ladder(ct, scopes=SCOPES, acfg=ACFG):
    inc, log = None, []
    for f in FAMILIES:
        b = _best(ct, f, scopes)
        if b is None: log.append(f'{f}: no admissible configuration'); continue
        if inc is None:
            inc = b; log.append(f"{f}: incumbent {b['mode']}/{b.scope} det {b.det_mean:.3f} alarm {b.alarm_mean:.4f}"); continue
        gain = [c - i for c, i in zip(b.det_by_seed, inc.det_by_seed)]
        win = all(g >= acfg['det_margin'] for g in gain) and b.alarm_mean <= inc.alarm_mean + acfg['alarm_slack']
        log.append(f"{f}: best {b['mode']}/{b.scope} det {b.det_mean:.3f} alarm {b.alarm_mean:.4f}; gains per seed {np.round(gain, 3).tolist()} -> "
                   + ('REPLACES ' if win else 'does not replace ') + f"{inc.family}")
        if win: inc = b
    return inc, log

def routing_table(dev, pi, acfg=ACFG):
    rows = []
    for (f, mo), t in dev[dev.eligible].groupby(['family', 'mode']):
        agg = t.groupby(['scope', 'seed']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean')).reset_index()
        gm = {s: agg[agg.scope == s].det.mean() for s in SCOPES}
        bg = 'G0' if gm['G0'] >= gm['G1'] else 'G1'
        dE = agg[agg.scope == 'E'].sort_values('seed').det.values; dG = agg[agg.scope == bg].sort_values('seed').det.values
        aE = agg[agg.scope == 'E'].alarm.mean(); aG = agg[agg.scope == bg].alarm.mean()
        p = pi[(pi.family == f) & (pi['mode'] == mo) & pi.scope.isin(['E', bg])]
        p = p.groupby(['scope', 'mac', 'inst']).hit.mean().reset_index()
        macs = sorted(t.mac.unique()); rng = np.random.default_rng(acfg['boot_seed'])
        H = {m: (p[(p.scope == 'E') & (p.mac == m)].sort_values('inst').hit.values, p[(p.scope == bg) & (p.mac == m)].sort_values('inst').hit.values) for m in macs}
        diffs = []
        for _ in range(acfg['boot_inst']):
            dd = []
            for m in macs:
                a, b = H[m]; i = rng.integers(0, len(a), len(a)); dd.append(a[i].mean() - b[i].mean())
            diffs.append(np.mean(dd))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        passed = bool(all((dE - dG) >= acfg['det_margin']) and lo > 0 and aE <= aG + acfg['alarm_slack'])
        rows.append(dict(family=f, mode=mo, det_E=float(dE.mean()), best_global=bg, det_global=float(dG.mean()), delta_by_seed=np.round(dE - dG, 3).tolist(),
                         ci95=[float(lo), float(hi)], alarm_E=float(aE), alarm_global=float(aG), routing_passes=passed))
    return pd.DataFrame(rows)

def decide(dev, pi, acfg=ACFG):
    ct = cfg_table(dev); sel, log = ladder(ct); rt = routing_table(dev, pi)
    out = dict(config_table=ct, ladder_log=log, routing_table=rt, selected=None if sel is None else sel.to_dict(), routing='no_admissible_detector', effective=None)
    if sel is None: return out
    if sel.scope == 'E':
        r = rt[(rt.family == sel.family) & (rt['mode'] == sel['mode'])].iloc[0]
        if r.routing_passes: out.update(routing='routing_kept', effective=sel.to_dict()); return out
        eff, log2 = ladder(ct, scopes=('G0', 'G1')); out['ladder_log_global_only'] = log2
        out.update(routing='routing_removed_tried_and_failed', effective=None if eff is None else eff.to_dict()); return out
    out.update(routing='routing_not_selected_global_is_best', effective=sel.to_dict()); return out

def detection_tiers(dev, eff, acfg=ACFG):
    if eff is None: return pd.DataFrame([dict(mac=m, device=n, det=np.nan, alarm=np.nan, n_dev_inst=k, detection_supported=False) for m, n, k in
                                         dev[['mac', 'device', 'n_dev_inst']].drop_duplicates().itertuples(index=False)])
    t = dev[(dev.family == eff['family']) & (dev['mode'] == eff['mode']) & (dev.scope == eff['scope'])]
    g = t.groupby(['mac', 'device']).agg(det=('det', 'mean'), alarm=('alarm_val', 'mean'), n_dev_inst=('n_dev_inst', 'first')).reset_index()
    g['detection_supported'] = (g.det >= 0.80) & (g.alarm <= 0.02) & (g.n_dev_inst >= acfg['min_dev_inst'])
    return g

# ---------------- identity (development split) ----------------
class MLPArm:
    def __init__(self, seed, clip=20.0): self.seed, self.clip = seed, clip
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = min(3000, max(np.bincount(y)))
        ii = np.concatenate([rng.choice(np.where(y == k)[0], n, replace=True) for k in np.unique(y)])
        self.sc = StandardScaler().fit(X[ii]); Z = np.clip(self.sc.transform(X[ii]), -self.clip, self.clip)
        self.m = MLPClassifier(hidden_layer_sizes=(128, 64), early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                               max_iter=200, random_state=self.seed).fit(Z, y[ii]); return self
    def predict_proba(self, X): return self.m.predict_proba(np.clip(self.sc.transform(X), -self.clip, self.clip))

def id_windows(D):
    W = {}
    for m, d in D.items():
        dtr, dval = d['dev_days']
        for part, days in (('train', dtr), ('calib', d['split']['calib']), ('val', dval)):
            W[(m, part)] = windows(d, days)
            if len(W[(m, part)][0]) == 0: raise ValueError(f"no identity windows for {d['name']} in {part}")
    return W

def run_identity_dev(D, W, cfg, seed, unknown, kinds=('RF', 'XGB', 'MLP')):
    from sklearn.metrics import f1_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.utils.class_weight import compute_sample_weight
    known = sorted(m for m in D if m not in unknown); kidx = {m: i for i, m in enumerate(known)}; K = len(known)
    def stack(part, devs):
        Xs, ys, dv, dd = [], [], [], []
        for m in devs:
            F, dy = W[(m, part)]; Xs.append(F); ys += [kidx.get(m, -1)] * len(F); dv += [m] * len(F); dd += list(dy)
        return np.vstack(Xs), np.array(ys), np.array(dv), np.array(dd)
    Xtr, ytr, _, _ = stack('train', known); Xca, yca, vca, _ = stack('calib', known); Xva, yva, vva, dva = stack('val', known)
    Xun, _, vun, _ = stack('val', unknown) if unknown else (np.zeros((0, Xtr.shape[1])), None, np.array([]), None)
    coh = np.array([D[m]['cohort'] for m in known])
    def pdv(P, vv):
        o = P.copy()
        for m in np.unique(vv):
            ii = np.where(vv == m)[0]; o[ii] = vote(P[ii])
        return o
    res = {}
    for kind in kinds:
        if kind == 'RF': mdl = RandomForestClassifier(n_estimators=cfg['rf_trees'], class_weight='balanced', n_jobs=-1, random_state=seed).fit(Xtr, ytr)
        elif kind == 'XGB':
            from xgboost import XGBClassifier
            mdl = XGBClassifier(n_estimators=cfg['xgb_trees'], max_depth=6, learning_rate=0.05, objective='multi:softprob', tree_method='hist',
                                n_jobs=-1, random_state=seed, num_class=K).fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else: mdl = MLPArm(seed).fit(Xtr, ytr)
        P = dict(ca=mdl.predict_proba(Xca), va=mdl.predict_proba(Xva), un=mdl.predict_proba(Xun) if len(Xun) else np.zeros((0, K)))
        for mode in ('single', 'vote3'):
            Q = P if mode == 'single' else dict(ca=pdv(P['ca'], vca), va=pdv(P['va'], vva), un=pdv(P['un'], vun) if len(Xun) else P['un'])
            thr = np.array([np.quantile(Q['ca'][yca == k, k], 0.10) for k in range(K)])
            pred = Q['va'].argmax(1); f1_dev = f1_score(yva, pred, average=None, labels=list(range(K)))
            acc = (Q['va'][np.arange(len(yva)), yva] >= thr[yva]).astype(float)
            rng = np.random.default_rng(seed)
            claim = np.array([rng.choice([k for k in range(K) if k != y]) for y in yva])
            rej = (Q['va'][np.arange(len(yva)), claim] < thr[claim]).astype(float)
            same = coh[claim] == coh[yva]
            hard = {}
            for A in range(K):
                mp = {B: Q['ca'][yca == B][:, A].mean() for B in range(K) if B != A}; hard[A] = max(mp, key=mp.get)
            rh = np.array([float(Q['va'][i, A] < thr[A]) for A in range(K) for i in np.where(yva == hard[A])[0]])
            ru = (Q['un'][:, :] < thr[None, :]).astype(float).ravel() if len(Xun) else np.array([])
            allrej = np.concatenate([rej, rh, ru])
            res[f'{kind}-{mode}'] = dict(
                f1_calib=float(f1_score(yca, Q['ca'].argmax(1), average='macro')), G1=float(f1_score(yva, pred, average='macro')),
                G2=float(np.mean([acc[yva == k].mean() for k in range(K)])), G3=float(rej.mean()), G4=float(rh.mean()),
                G5=float(ru.mean()) if len(ru) else None, G6=float(1 - allrej.mean()),
                rej_same_cohort=float(rej[same].mean()) if same.any() else None, rej_cross_cohort=float(rej[~same].mean()) if (~same).any() else None,
                per_device={known[k]: dict(f1=float(f1_dev[k]), acceptance=float(acc[yva == k].mean()), claim_rejection=float(rej[claim == k].mean()) if (claim == k).any() else None)
                            for k in range(K)}, n_val=int(len(yva)))
    return res

def identity_summary(held, closed, seeds, tol=0.0):
    """held/closed: {seed: {arm: metrics}}. Returns admission + primary arm + per-device decision (closed-set, primary arm, seed means)."""
    arms = sorted(held[seeds[0]])
    g2 = {a: [held[s][a]['G2'] for s in seeds] for a in arms}; g6 = {a: [held[s][a]['G6'] for s in seeds] for a in arms}
    simple = [a for a in arms if not a.startswith('MLP')]; mlp = [a for a in arms if a.startswith('MLP')]
    admitted, note = list(simple), []
    if mlp:
        for a in mlp:
            mode = a.split('-')[1]; ref = [x for x in simple if x.endswith(mode)]
            best = max(ref, key=lambda x: np.mean(g2[x]))
            ok = all(g2[a][i] - g2[best][i] >= 0.05 for i in range(len(seeds))) and all(g6[a][i] - g6[best][i] <= 0.01 for i in range(len(seeds)))
            note.append(f'{a} vs {best}: G2 gains per seed {np.round(np.array(g2[a]) - np.array(g2[best]), 3).tolist()} -> ' + ('ADMITTED' if ok else 'not admitted'))
            if ok: admitted.append(a)
    primary = max(admitted, key=lambda a: (round(np.mean([held[s][a]['f1_calib'] for s in seeds]), 12), -arms.index(a)))
    devs = closed[seeds[0]][primary]['per_device']; dec = []
    for m in devs:
        f1 = np.mean([closed[s][primary]['per_device'][m]['f1'] for s in seeds]); ac = np.mean([closed[s][primary]['per_device'][m]['acceptance'] for s in seeds])
        cr = [closed[s][primary]['per_device'][m]['claim_rejection'] for s in seeds if closed[s][primary]['per_device'][m]['claim_rejection'] is not None]
        crm = float(np.mean(cr)) if cr else np.nan
        dec.append(dict(mac=m, f1=float(f1), acceptance=float(ac), claim_rejection=crm, identity_supported=bool(f1 >= 0.80 and ac >= 0.80 and crm >= 0.95)))
    return dict(primary=primary, admitted=admitted, mlp_admission=note, device_identity=pd.DataFrame(dec),
                held_gates={a: {k: float(np.mean([held[s][a][k] for s in seeds])) for k in ('G1', 'G2', 'G3', 'G4', 'G5', 'G6')} for a in arms})

def assign_tiers(D, ident, dtier):
    t = ident['device_identity'].merge(dtier[['mac', 'det', 'alarm', 'n_dev_inst', 'detection_supported']], on='mac', how='left')
    t['device'] = t.mac.map({m: D[m]['name'] for m in D})
    def tier(r):
        if not r.identity_supported: return 'UNREGISTERED'
        return 'VERIFIED' if bool(r.detection_supported) else 'IDENTIFIED_UNVALIDATED'
    t['tier'] = t.apply(tier, axis=1)
    return t[['device', 'mac', 'tier', 'f1', 'acceptance', 'claim_rejection', 'identity_supported', 'det', 'alarm', 'n_dev_inst', 'detection_supported']]

## Download data

In [ ]:
import os, zipfile, shutil, requests, glob, hashlib, time
OUT = '/kaggle/working/n2_phaseA'; os.makedirs(OUT, exist_ok=True)
DATA = '/kaggle/working/unsw_attack'; os.makedirs(DATA, exist_ok=True)
BASE = 'https://iotanalytics.unsw.edu.au/anomaly-data/'
hashes = {}
for n in ['flowdata.zip', 'annotations.zip']:
    p = (glob.glob(f'/kaggle/input/**/{n}', recursive=True) or [None])[0]
    if p is None:
        p = f'{DATA}/{n}'
        with requests.get(BASE + n, stream=True, timeout=600, headers={'User-Agent': 'ris-n2'}) as r:
            r.raise_for_status()
            with open(p, 'wb') as fh: shutil.copyfileobj(r.raw, fh)
    hashes[n] = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    with zipfile.ZipFile(p) as z: z.extractall(f'{DATA}/{n[:-4]}')
print(hashes); json.dump(hashes, open(f'{OUT}/data_hashes.json', 'w'), indent=1)

## Load and development split

In [ ]:
D = load_all(f'{DATA}/flowdata', f'{DATA}/annotations'); prepare_dev(D)
assert all('m_test' not in d for d in D.values())
info = pd.DataFrame([dict(device=d['name'], cohort=d['cohort'], instances=len(d['inst']), dev_instances=len(d['dev_inst']), eligible=d['eligible'],
                          dtrain_min=int(d['m_dtrain'].sum()), dval_min=int(d['m_dval'].sum()), calib_min=int(d['m_calib'].sum())) for d in D.values()])
print(info.to_string(index=False)); info.to_csv(f'{OUT}/dev_split_summary.csv', index=False)

## A.1 Identity: held-out run (gates) and closed-set run (per-device decisions)

In [ ]:
t0 = time.time(); SEEDS = ACFG['seeds']
W = id_windows(D); unk = pick_unknown(D, CFG['split_seed'])
print('held-out identity devices:', [D[m]['name'] for m in unk])
held = {s: run_identity_dev(D, W, CFG, s, unk) for s in SEEDS}
closed = {s: run_identity_dev(D, W, CFG, s, [], kinds=('RF', 'XGB', 'MLP')) for s in SEEDS}
isum = identity_summary(held, closed, SEEDS)
print('primary arm:', isum['primary'], '| admitted:', isum['admitted']); print('\n'.join(isum['mlp_admission']))
print(pd.DataFrame(isum['held_gates']).T.round(3).to_string())
print(isum['device_identity'].assign(device=lambda t: t.mac.map({m: D[m]['name'] for m in D})).round(3).to_string(index=False))
print('same/cross-cohort random-mismatch rejection (held-out run, primary arm, seed 11):', held[11][isum['primary']]['rej_same_cohort'], held[11][isum['primary']]['rej_cross_cohort'])
json.dump(dict(held=held, closed=closed), open(f'{OUT}/identity_runs.json', 'w'), indent=1, default=float)
isum['device_identity'].to_csv(f'{OUT}/identity_device_decision.csv', index=False)
print('identity done', round(time.time() - t0), 's')

## A.2 Detection: families x modes x scopes x seeds (checkpointed)

In [ ]:
devs, pis = [], []
for fam in FAMILIES:
    for s in SEEDS:
        fp = f'{OUT}/det_{fam}_{s}.csv'
        if os.path.exists(fp) and os.path.exists(fp.replace('det_', 'pi_')):
            devs.append(pd.read_csv(fp)); pis.append(pd.read_csv(fp.replace('det_', 'pi_'))); continue
        t0 = time.time(); a, b = run_family(D, fam, s, ACFG, CFG)
        a.to_csv(fp, index=False); b.to_csv(fp.replace('det_', 'pi_'), index=False); devs.append(a); pis.append(b)
        print(fam, 'seed', s, round(time.time() - t0), 's', flush=True)
dev = pd.concat(devs, ignore_index=True); pi = pd.concat(pis, ignore_index=True)
dev.to_csv(f'{OUT}/detection_dev_all.csv', index=False); pi.to_csv(f'{OUT}/detection_instances_all.csv', index=False)

## A.2 Supervised reference (leave-one-family-out, seed 11, not selectable)

In [ ]:
sup, sup_inst = run_supervised(D, 11, ACFG, CFG)
if hasattr(sup, 'to_csv') and len(sup):
    sup.to_csv(f'{OUT}/supervised_reference.csv', index=False); sup_inst.to_csv(f'{OUT}/supervised_reference_instances.csv', index=False)
    print(sup.round(3).to_string(index=False)); print('macro det_lofo over eligible devices:', round(float(sup[sup.eligible].det_lofo.mean()), 3), '| mean alarm:', round(float(sup[sup.eligible].alarm_val.mean()), 4))
else: print(sup)

## Decision: ladder, routing test, device tiers

In [ ]:
out = decide(dev, pi)
ct = out['config_table']; print(ct.drop(columns=['det_by_seed', 'alarm_by_seed']).round(4).to_string(index=False))
print('\n'.join(out['ladder_log'])); print(out.get('ladder_log_global_only', ''))
print(out['routing_table'].round(3).to_string(index=False))
print('ROUTING:', out['routing']); print('EFFECTIVE DETECTOR:', out['effective'] and {k: out['effective'][k] for k in ('family', 'mode', 'scope', 'det_mean', 'alarm_mean')})
dtier = detection_tiers(dev, out['effective']); tiers = assign_tiers(D, isum, dtier)
print(tiers.round(3).to_string(index=False)); tiers.to_csv(f'{OUT}/device_tiers_phaseA.csv', index=False)
ct.to_csv(f'{OUT}/config_table.csv', index=False); out['routing_table'].to_csv(f'{OUT}/routing_table.csv', index=False)

## Save

In [ ]:
final = dict(protocol_sha256=sha, chain_block=blk['hash'], data_hashes=hashes, acfg={k: (list(v) if isinstance(v, tuple) else v) for k, v in ACFG.items()},
             identity=dict(primary=isum['primary'], admitted=isum['admitted'], mlp_admission=isum['mlp_admission'], held_gates=isum['held_gates']),
             detection=dict(selected=out['selected'], effective=out['effective'], ladder_log=out['ladder_log'], routing=out['routing']),
             tiers=tiers.to_dict('records'))
json.dump(final, open(f'{OUT}/phaseA_results.json', 'w'), indent=1, default=str)
shutil.make_archive('/kaggle/working/n2_phaseA_results', 'zip', OUT)
print('TIERS:', tiers.groupby('tier').device.apply(list).to_dict()); print('ROUTING:', out['routing'], '| PRIMARY IDENTITY ARM:', isum['primary'])